In [ ]:
import lightgbm as lgm
import pandas as pd
import numpy as np
import gc
import os
import pickle
import joblib

In [2]:
def rmsse_from_scale(y_true, y_pred, scale):
    if scale is None or np.isnan(scale) or scale == 0:
        return np.nan
    mse = np.mean((y_true - y_pred) ** 2)
    return np.sqrt(mse / scale)

In [3]:
#4
#lgm_params={'subsample': 0.8, 'num_leaves': 100, 'n_estimators': 750, 'min_data_in_leaf': 250, 'max_depth': 25, 'max_bin': 200, 'learning_rate': 0.02, 'colsample_bytree': 0.85, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
#0.7750640935226275,0.8418748007662326
#3
lgm_params={'subsample': 0.6, 'num_leaves': 300, 'n_estimators': 1000, 'min_data_in_leaf': 100, 'max_depth': 15, 'max_bin': 200, 'learning_rate': 0.01, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
#0.7736663771184371,0.8402693400581798
#5
#lgm_params={'subsample': 0.6, 'num_leaves': 300, 'n_estimators': 1500, 'min_data_in_leaf': 100, 'max_depth': 10, 'max_bin': 200, 'learning_rate': 0.01, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
#0.7743342446182332

In [4]:
#1
#lgm_params= {'subsample': 0.75, 'num_leaves': 150, 'n_estimators': 500, 'min_data_in_leaf': 200, 'max_depth': 15, 'max_bin': 100, 'learning_rate': 0.01, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1} 
#0.7745779371445467,0.8421406474970609
#2
#lgm_params={'subsample': 0.8, 'num_leaves': 70, 'n_estimators': 750, 'min_data_in_leaf': 200, 'max_depth': 25, 'max_bin': 200, 'learning_rate': 0.05, 'colsample_bytree': 0.85, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
#0.776956292766264,0.8435859020965982

In [5]:
SHIFT_WINDOW_PAIRS = [(s, w) for s in (1, 7) for w in (7, 14, 30, 60)]
DYNAMIC_COLS = [f"rolling_mean_{s}_{w}" for s, w in SHIFT_WINDOW_PAIRS]

OTHER = r"D:\M5 Forecasting"

In [6]:
sales = pd.read_pickle(OTHER + r"\Sale-Store\sales-CA_1.pkl")
sales.isna().sum()

store_id                       0
item_id                        0
sell_price                     0
price_max                      0
price_min                      0
price_mean                     0
price_norm                     0
price_lag_w1               21343
price_lag_m6              554716
price_momentum_w1          21343
price_momentum_m6         554716
Date                           0
sales                          0
wday                           0
month                          0
year                           0
event_name_1                   0
event_type_1                   0
event_name_2                   0
event_type_2                   0
snap_CA                        0
day_month                      0
week_month                     0
week_end                       0
dept_id_enc_mean               0
item_id_enc_mean               0
item_id_enc_std                0
state_item_id_enc_mean         0
cat_id_enc_std                 0
state_dept_id_enc_std          0
rolling_so

In [ ]:
OTHER = r"D:\M5 Forecasting"
DATE = 1913

D_SHIFT_LIST = [1, 7]
D_WINDOW_LIST = [7, 14, 30]
DYNAMIC_COLS = [f"rolling_mean_{s}_{w}" for s in D_SHIFT_LIST for w in D_WINDOW_LIST]

os.makedirs(OTHER + r"\Models", exist_ok=True)
os.makedirs(OTHER + r"\Results", exist_ok=True)

rmsse = pd.read_pickle(OTHER + rf"\Sale-Store\rmsselookup.pkl")

numeric_cols = [
    'dept_id_enc_mean', 'dept_id_enc_std',
    'item_id_enc_mean', 'item_id_enc_std',
    'state_item_id_enc_mean', 'state_item_id_enc_std',
    'cat_id_enc_mean', 'cat_id_enc_std',
    'state_dept_id_enc_mean', 'state_dept_id_enc_std',
    'store_dept_id_enc_mean', 'store_dept_id_enc_std',
    'store_cat_id_enc_std',
    'store_item_id_enc_mean', 'store_item_id_enc_std',
    'sales_lag_29', 'sales_lag_30', 'sales_lag_35', 'sales_lag_42', 'sales_lag_58',
    'sellingTrend',
]

STORES = ['CA_1','CA_2','CA_3','CA_4','TX_1','TX_2','TX_3','WI_1','WI_2','WI_3']

all_item_rmsse = []
all_preds_global = []
store_weights = {}

for i in STORES:
    print("TRAINING AND TESTING ", i)
    testdata = pd.read_pickle(OTHER + rf"\Predict-Store\sales-{i}.pkl")
    sales = pd.read_pickle(OTHER + rf"\Sale-Store\sales-{i}.pkl")
    actual = pd.read_pickle(OTHER + rf"\Predict-Store\sales-correct-{i}.pkl")
    gc.collect()
    scale_lookup = rmsse.get(i, {})
    store_weights[i] = actual["sales"].sum()
    sales["sales"] = pd.to_numeric(sales["sales"])
    for col in numeric_cols:
        if col in sales.columns:
            sales[col] = pd.to_numeric(sales[col])
        if col in testdata.columns:
            testdata[col] = pd.to_numeric(testdata[col])
    feature_cols = [c for c in sales.columns if c not in ("sales", "store_id")]
    model = lgm.LGBMRegressor(**lgm_params)
    model.fit(
        sales[feature_cols], sales["sales"],
        eval_set=[(sales[feature_cols], sales["sales"])],   
        eval_metric="rmse",
        callbacks=[lgm.early_stopping(30), lgm.log_evaluation(50)]
    )
    joblib.dump(model, OTHER + rf"\Models\{i}_recursive_model.pkl")
    print(f"  saved model -> Models\\{i}_recursive_model.pkl")
    '''
    date = DATE
    last_day = sales[sales["Date"] == date][["item_id"] + DYNAMIC_COLS].set_index("item_id")
    hist = sales[sales["Date"] > date - 100][["item_id", "Date", "sales"]]   # widened to 100, per reference kernel
    future = testdata[["item_id", "Date"]].copy()
    future["sales"] = np.nan
    buf = pd.concat([hist, future])
    del hist, future

    store_day_preds = []
    for day in range(1, 29):
        current_date = date + day
        x = testdata[testdata["Date"] == current_date].set_index("item_id", drop=False).copy()

        missing = x.index.difference(last_day.index)
        if len(missing) > 0:
            x = x.drop(index=missing)
        g = buf.groupby("item_id")["sales"]
        for d_shift in D_SHIFT_LIST:
            for d_window in D_WINDOW_LIST:
                col_name = f"rolling_mean_{d_shift}_{d_window}"
                feat = g.transform(lambda s: s.shift(d_shift).rolling(d_window).mean())
                temp = buf.assign(**{col_name: feat})
                temp = temp[temp["Date"] == current_date].set_index("item_id")[col_name]
                x[col_name] = temp.reindex(x.index).values

        preds = model.predict(x[feature_cols])
        preds = np.clip(preds, 0, None)
        preds = np.round(preds, 0).astype(float)
        x["sales"] = preds

        store_day_preds.append(x[["item_id", "store_id", "Date", "sales"]].reset_index(drop=True))

        buf.loc[buf["Date"] == current_date, "sales"] = (
            buf.loc[buf["Date"] == current_date, "item_id"].map(x.set_index("item_id")["sales"]).to_numpy()
        )

    preds_28 = pd.concat(store_day_preds, ignore_index=True)

    merged = preds_28.merge(actual, on=["item_id", "store_id", "Date"], suffixes=("_pred", "_true"))
    item_rmsse = {}
    for item in merged["item_id"].unique():
        g = merged[merged["item_id"] == item]
        y_true = g["sales_true"].to_numpy()
        y_pred = g["sales_pred"].to_numpy()
        scale = scale_lookup.get(item, np.nan)
        item_rmsse[item] = rmsse_from_scale(y_true, y_pred, scale)

    item_rmsse = pd.Series(item_rmsse, name="RMSSE")
    print(f"{i} store RMSSE ", item_rmsse.mean())
    print(item_rmsse.describe())

    all_item_rmsse.append(item_rmsse.rename(i))
    all_preds_global.append(preds_28.assign(store_id=i))
    print()

    del model, sales, testdata, actual, buf, preds_28
    gc.collect()

full_rmsse = pd.concat(all_item_rmsse, axis=1)
full_preds = pd.concat(all_preds_global, ignore_index=True)

full_rmsse.to_pickle(OTHER + r"\Results\recursive_full_rmsse.pkl")
full_preds.to_pickle(OTHER + r"\Results\recursive_full_preds.pkl")

with open(OTHER + r"\Results\recursive_store_weights.pkl", "wb") as f:
    pickle.dump(store_weights, f)

total_weight = sum(store_weights.values())
weighted_rmsse = sum((store_weights[s] / total_weight) * full_rmsse[s].mean() for s in STORES)

print("\n===== FINAL SUMMARY =====")
print(full_rmsse.mean().sort_values())
print(f"\nUnweighted mean RMSSE across stores: {full_rmsse.mean().mean():.4f}")
print(f"Weighted RMSSE (by store sales volume): {weighted_rmsse:.4f}")'''

TRAINING AND TESTING  CA_1
Training until validation scores don't improve for 30 rounds
[50]	valid_0's rmse: 3.73156	valid_0's tweedie: 16.3227
[100]	valid_0's rmse: 3.04583	valid_0's tweedie: 16.039
[150]	valid_0's rmse: 2.66754	valid_0's tweedie: 15.9395
[200]	valid_0's rmse: 2.48541	valid_0's tweedie: 15.8968
[250]	valid_0's rmse: 2.39791	valid_0's tweedie: 15.8763
[300]	valid_0's rmse: 2.35172	valid_0's tweedie: 15.8651
[350]	valid_0's rmse: 2.32272	valid_0's tweedie: 15.8582
[400]	valid_0's rmse: 2.30268	valid_0's tweedie: 15.8535
[450]	valid_0's rmse: 2.28694	valid_0's tweedie: 15.85
[500]	valid_0's rmse: 2.27593	valid_0's tweedie: 15.8471
[550]	valid_0's rmse: 2.26556	valid_0's tweedie: 15.8445
[600]	valid_0's rmse: 2.25558	valid_0's tweedie: 15.8423
[650]	valid_0's rmse: 2.2468	valid_0's tweedie: 15.8403
[700]	valid_0's rmse: 2.23776	valid_0's tweedie: 15.8386
[750]	valid_0's rmse: 2.23171	valid_0's tweedie: 15.8373
[800]	valid_0's rmse: 2.22607	valid_0's tweedie: 15.836
[850]	

In [ ]:


OTHER = r"D:\M5 Forecasting"
SUFFIX = "_validation"
HORIZON = 28

full_preds = pd.read_pickle(OTHER + r"\Results\recursive_full_preds.pkl")


def to_submission_format(preds_df, suffix, horizon=HORIZON):
    df = preds_df.copy()
    df["id"] = df["item_id"].astype(str) + "_" + df["store_id"] + suffix

    day_map = {d: f"F{rank+1}" for rank, d in enumerate(sorted(df["Date"].unique()))}
    df["F"] = df["Date"].map(day_map)

    wide = df.pivot(index="id", columns="F", values="sales")
    f_cols = [f"F{i}" for i in range(1, horizon + 1)]
    wide = wide.reindex(columns=f_cols).reset_index()
    return wide

submission_validation = to_submission_format(full_preds, SUFFIX)



sample_submission = pd.read_csv(r"C:\Users\Shree\Desktop\CODE\M5 Forecasting\Data\sample_submission.csv")
required_ids = sample_submission[sample_submission["id"].str.endswith(SUFFIX)][["id"]]

final_validation = required_ids.merge(submission_validation, on="id", how="left")
f_cols = [f"F{i}" for i in range(1, HORIZON + 1)]
final_validation[f_cols] = final_validation[f_cols].fillna(0)

missing = final_validation[final_validation.drop(columns="id").sum(axis=1) == 0]
print(f"{len(missing)} ids had no prediction and were filled with 0")

final_validation.to_csv(OTHER + r"\Submission\submission_validation.csv", index=False)
print("Saved submission_validation.csv —", len(final_validation), "rows")
final_validation.head()


In [ ]:
rmsse['TX_1']["HOUSEHOLD_1_032"]   # from the rmsse dict for TX_1 #np.float64(0.0014992503748125937)
#THE ABOVE VALUE IS CLOSE TO 0 HENC THE RMSSE SHOOT UP AS 61 IN THE TX_1

In [ ]:
#item_rmsse.describe()
#item_rmsse.sort_values(ascending=False).head(10)   # which items are worst 0.7282552118429804

In [ ]:
lgb_params={'subsample': 0.6, 'num_leaves': 50, 'n_estimators': 750, 'min_data_in_leaf': 300, 'max_depth': 25, 'max_bin': 200, 'learning_rate': 0.01, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}

In [ ]:
import lightgbm as lgm
import pandas as pd
import numpy as np
import gc
import os
import joblib
import pickle
OTHER = r"D:\M5 Forecasting"
rmsse = pd.read_pickle(OTHER + rf"\Sale-Store\rmsselookup.pkl")
STORES = ['CA_1','CA_2','CA_3','CA_4','TX_1','TX_2','TX_3','WI_1','WI_2','WI_3']
os.makedirs(OTHER + r"\Models", exist_ok=True)
os.makedirs(OTHER + r"\Results", exist_ok=True)
numeric_cols = [
        'dept_id_enc_mean', 'dept_id_enc_std',
    'item_id_enc_mean', 'item_id_enc_std',
    'state_item_id_enc_mean', 'state_item_id_enc_std',
    'cat_id_enc_mean', 'cat_id_enc_std',
    'state_dept_id_enc_mean', 'state_dept_id_enc_std',
    'store_dept_id_enc_mean', 'store_dept_id_enc_std',
    'store_cat_id_enc_std',
    'store_item_id_enc_mean', 'store_item_id_enc_std',
    'sales_lag_29', 'sales_lag_30', 'sales_lag_35', 'sales_lag_42', 'sales_lag_58',
    'sellingTrend',
]

all_item_rmssenon = []
all_predsnon = []
store_weights_non = {}
for i in STORES:
    print("TRAINING AND TESTING ", i)
    testdata = pd.read_pickle(OTHER + rf"\Predict-Store\sales-{i}.pkl")
    sales = pd.read_pickle(OTHER + rf"\Sale-Store\sales-{i}.pkl")
    sales = sales.drop(columns=[c for c in DYNAMIC_COLS if c in sales.columns])
    actual = pd.read_pickle(OTHER + rf"\Predict-Store\sales-correct-{i}.pkl")
    gc.collect()
    scale_lookup = rmsse.get(i, {})
    store_weights_non[i] = actual["sales"].sum()
    for col in numeric_cols:
        if col in sales.columns:
            sales[col] = pd.to_numeric(sales[col])
        if col in testdata.columns:
            testdata[col] = pd.to_numeric(testdata[col])
    sales["sales"] = pd.to_numeric(sales["sales"])
    feature_cols = [c for c in sales.columns if c not in ("sales", "store_id")]
    model = lgm.LGBMRegressor(**lgb_params)
    model.fit(sales[feature_cols], sales["sales"])
    preds_vals = model.predict(testdata[feature_cols])   
    preds_vals = np.clip(preds_vals, 0, None)
    preds_vals = np.round(preds_vals).astype(float)
    preds_df = testdata[["item_id", "store_id", "Date"]].copy()
    preds_df["sales"] = preds_vals
    merged = preds_df.merge(actual, on=["item_id", "store_id", "Date"], suffixes=("_pred", "_true"))
    item_rmsse = {}
    for item in merged["item_id"].unique():
        g = merged[merged["item_id"] == item]
        y_true = g["sales_true"].to_numpy()
        y_pred = g["sales_pred"].to_numpy()
        scale = scale_lookup.get(item, np.nan)
        item_rmsse[item] = rmsse_from_scale(y_true, y_pred, scale)
    item_rmsse = pd.Series(item_rmsse, name="RMSSE")
    joblib.dump(model, OTHER + rf"\Models\{i}_nonrecursive_model.pkl")
    print(f"{i} store RMSSE ", item_rmsse.mean())
    print(item_rmsse.describe())
    all_item_rmssenon.append(item_rmsse.rename(i))
    all_predsnon.append(preds_df.assign(store_id=i))
    print()
    del model, sales, testdata, actual, merged, item_rmsse
    gc.collect()
full_rmsse_non = pd.concat(all_item_rmssenon, axis=1)
full_preds_non = pd.concat(all_predsnon, ignore_index=True)
full_rmsse_non.to_pickle(OTHER + r"\Results\nonrecursive_full_rmsse.pkl")
full_preds_non.to_pickle(OTHER + r"\Results\nonrecursive_full_preds.pkl")
with open(OTHER + r"\Results\nonrecursive_store_weights.pkl", "wb") as f:
    pickle.dump(store_weights_non, f)
total_weight = sum(store_weights_non.values())
weighted_rmsse_non = sum((store_weights_non[s] / total_weight) * full_rmsse_non[s].mean()for s in STORES)
print("\n===== NON-RECURSIVE FINAL SUMMARY =====")
print(full_rmsse_non.mean().sort_values())
print(f"\nUnweighted mean RMSSE: {full_rmsse_non.mean().mean():.4f}")
print(f"Weighted RMSSE: {weighted_rmsse_non:.4f}")
nonrecursive_preds = pd.read_pickle(OTHER + r"\Results\nonrecursive_full_preds.pkl")

In [ ]:

recursive_preds = pd.read_pickle(OTHER + r"\Results\recursive_full_preds.pkl")
nonrecursive_preds = pd.read_pickle(OTHER + r"\Results\nonrecursive_full_preds.pkl")
actual_all = []
for store in ['CA_1','CA_2','CA_3','CA_4','TX_1','TX_2','TX_3','WI_1','WI_2','WI_3']:
    a = pd.read_pickle(OTHER + rf"\Predict-Store\sales-correct-{store}.pkl")
    actual_all.append(a)
actual = pd.concat(actual_all, ignore_index=True)
rmsse_scale = pd.read_pickle(OTHER + r"\Sale-Store\rmsselookup.pkl")
ensemble = recursive_preds.merge(nonrecursive_preds,on=["item_id", "store_id", "Date"],suffixes=("_rec", "_nonrec"))
print("recursive rows:", len(recursive_preds))
print("nonrecursive rows:", len(nonrecursive_preds))
print("merged rows:", len(ensemble))

ensemble["sales"] = (ensemble["sales_rec"] + ensemble["sales_nonrec"]) / 2
ensemble["sales"] = np.round(ensemble["sales"]).clip(lower=0)
def rmsse_from_scale(y_true, y_pred, scale):
    if scale is None or np.isnan(scale) or scale == 0:
        return np.nan
    mse = np.mean((y_true - y_pred) ** 2)
    return np.sqrt(mse / scale)

merged = ensemble.merge(actual, on=["item_id", "store_id", "Date"], suffixes=("", "_true"))

results = []
for store in merged["store_id"].unique():
    store_scale = rmsse_scale.get(store, {})
    store_df = merged[merged["store_id"] == store]

    for item, g in store_df.groupby("item_id"):
        scale = store_scale.get(item, np.nan)
        score = rmsse_from_scale(g["sales_true"].to_numpy(), g["sales"].to_numpy(), scale)
        if not np.isnan(score):
            results.append({"store_id": store, "item_id": item, "rmsse": score})

ensemble_rmsse = pd.DataFrame(results)



full_rmsse_rec = pd.read_pickle(OTHER + r"\Results\recursive_full_rmsse.pkl")
full_rmsse_nonrec = pd.read_pickle(OTHER + r"\Results\nonrecursive_full_rmsse.pkl")

print("\n===== COMPARISON =====")
print(f"Recursive alone:     {full_rmsse_rec.mean().mean():.4f}")
print(f"Non-recursive alone: {full_rmsse_nonrec.mean().mean():.4f}")
print(f"Ensemble (average):  {ensemble_rmsse['rmsse'].mean():.4f}")

per_store = ensemble_rmsse.groupby("store_id")["rmsse"].mean().sort_values()
print("\nEnsemble RMSSE per store:")
print(per_store)



ensemble[["item_id","store_id","Date","sales"]].to_pickle(OTHER + r"\Results\ensemble_preds.pkl")
ensemble_rmsse.to_pickle(OTHER + r"\Results\ensemble_rmsse.pkl")

print("\nSaved: Results\\ensemble_preds.pkl, Results\\ensemble_rmsse.pkl")

In [ ]:
merged_for_fit = recursive_preds.merge(
    nonrecursive_preds, on=["store_id","item_id","Date"], suffixes=("_rec","_nonrec")
).merge(
    actual, on=["store_id","item_id","Date"]
)

fit_rows = merged_for_fit[merged_for_fit["Date"] <= date]
train = pd.DataFrame({
    "RECURSIVE": fit_rows["sales_rec"],
    "NONRECURSIVE": fit_rows["sales_nonrec"],
    "ACTUAL": fit_rows["sales"]
})
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(train[["RECURSIVE","NONRECURSIVE"]], train["ACTUAL"])

X = pd.DataFrame({
    "RECURSIVE": merged_for_fit["sales_rec"],
    "NONRECURSIVE": merged_for_fit["sales_nonrec"]
})
ensemble_sales = np.round(np.clip(model.predict(X), 0, None))

ensemble = merged_for_fit[["store_id","item_id","Date"]].copy()
ensemble["sales"] = ensemble_sales